# Prepare stimulus labels and splits

Run from the repository root. Run the function cell and then the cell for the dataset you have locally. Existing historical splits are preserved and validated; absent splits use a fixed seed. Labels describe stimulus conditions, not participant responses.


In [ ]:
from pathlib import Path
import random
import numpy as np

def prepare_lists(dataset, seed=1234):
    if not Path('cfg').is_dir():
        raise RuntimeError('Run from the repository root.')
    files = sorted(p.name for p in Path(f'{dataset}_data').glob('*.wav'))
    if not files:
        raise FileNotFoundError('Generate the audio first.')
    positive, negative = ('sig', 'nosig') if dataset == 'sim' else ('sigwf', 'nswf')
    labels = {}
    for name in files:
        if name.startswith(positive):
            labels[name] = 1
        elif name.startswith(negative):
            labels[name] = 0
        else:
            raise ValueError(f'Unexpected filename: {name}')
    if set(labels.values()) != {0, 1}:
        raise ValueError('Both classes must be present.')
    output = Path('data_lists')
    output.mkdir(exist_ok=True)
    train_path = output / f'{dataset}_train.scp'
    test_path = output / f'{dataset}_test.scp'
    if train_path.exists() != test_path.exists():
        raise ValueError('Only one split exists; restore the matching pair.')
    existing = train_path.exists()
    if existing:
        train = train_path.read_text().splitlines()
        test = test_path.read_text().splitlines()
    else:
        shuffled = files.copy()
        random.Random(seed).shuffle(shuffled)
        boundary = int(0.8 * len(shuffled))
        train, test = shuffled[:boundary], shuffled[boundary:]
    if not train or not test:
        raise ValueError('Both splits must be nonempty.')
    if len(set(train)) != len(train) or len(set(test)) != len(test):
        raise ValueError('Duplicate filenames in a split.')
    if set(train) & set(test):
        raise ValueError('Train/test overlap.')
    if (set(train) | set(test)) != set(files):
        raise ValueError('Splits do not match available audio; keep historical data separate.')
    if not existing:
        train_path.write_text('\n'.join(train) + '\n', encoding='utf-8')
        test_path.write_text('\n'.join(test) + '\n', encoding='utf-8')
    np.save(output / f'{dataset}_labels.npy', labels)
    print(f'{dataset}: train={len(train)}, test={len(test)}, existing split preserved={existing}')


## Synthetic data


In [ ]:
prepare_lists('sim')


## Experiment data (local audio required)


In [ ]:
prepare_lists('exp')
